In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
loader = TextLoader("fpt_policy.txt", encoding="utf-8")
documents = loader.load()

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)

In [5]:
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(texts, gemini_embeddings, collection_name="fpt_policy")

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [7]:
@tool
def search_fpt_policy(query: str) -> str:
    """
    Hãy sử dụng công cụ này BẮT BUỘC khi bạn cần tìm kiếm thông tin về quy định, 
    chính sách, nội quy, giờ giấc, lương thưởng hoặc bất kỳ vấn đề nhân sự nào của FPT.
    """
   
    results = retriever.invoke(query) 
    
    return "\n\n---\n\n".join([result.page_content for result in results])

In [8]:
SYSTEM_PROMPT = """
Bạn là một trợ lý Nhân sự (HR) mẫn cán của tập đoàn FPT. 
Nhiệm vụ của bạn là giải đáp thắc mắc của nhân viên về các quy định nội bộ.

Quy tắc bắt buộc:
1. LUÔN LUÔN sử dụng công cụ 'search_fpt_policy' trước khi trả lời bất kỳ câu hỏi nào về quy định công ty.
2. Tuyệt đối không tự bịa thông tin. Dựa 100% vào dữ liệu tìm được.
3. Hãy trả lời ngắn gọn, lịch sự.
"""
# mmodel = init_chat_model(model="openai:gpt-4.1-mini", temperature=0.0)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.0
)
agent = create_agent(
    model=llm, 
    tools=[search_fpt_policy], 
    system_prompt=SYSTEM_PROMPT).with_config(max_iterations=3)


In [9]:
question = "Nếu mình đi làm muộn 50 phút thì công ty phạt bao nhiêu tiền?"

print(f"🧑 User hỏi: {question}\n")
print("🔄 Agent đang suy nghĩ và tìm kiếm...\n")
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": question}
        ]
    }
)

final_answer = result["messages"][-1].content

if isinstance(final_answer, list):
    # Lấy giá trị 'text' của phần tử đầu tiên
    print(final_answer[0].get('text', ''))
else:
    print(final_answer)



🧑 User hỏi: Nếu mình đi làm muộn 50 phút thì công ty phạt bao nhiêu tiền?

🔄 Agent đang suy nghĩ và tìm kiếm...



ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 38.780756898s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
// 4 bước thực thi ngầm của retriever.invoke(query)
// Giả sử query mà Agent truyền vào là chuỗi chữ: "phạt đi muộn 5 phút".

// Bước 1: Nhúng (Embedding) câu hỏi
// Nó không mang nguyên dòng chữ "phạt đi muộn 5 phút" đi tìm kiếm. Việc đầu tiên nó làm là âm thầm gọi lại cỗ máy gemini-embedding-001 (mà bạn đã nạp vào VectorDB từ trước). Cỗ máy này sẽ dịch câu hỏi tiếng Việt đó thành một mảng số học (Vector tọa độ) gồm hàng ngàn chiều.

// Ví dụ: [0.12, -0.84, 0.55, ...]

// Bước 2: Quét và Đo khoảng cách (Similarity Search)
// Nó cầm cái Vector tọa độ vừa dịch được, lao thẳng vào kho dữ liệu ChromaDB. Tại đây, nó dùng các công thức toán học (thường là Cosine Similarity - tính toán góc giữa 2 vector) để đo khoảng cách từ điểm tọa độ của "câu hỏi" đến TẤT CẢ các điểm tọa độ của các "đoạn văn bản" đang nằm trong kho.

// Bước 3: Lọc Top K (Top-K Selection)
// Nó không lấy hết dữ liệu. Nhớ lại cái thiết lập search_kwargs={"k": 3} của bạn chứ? Lúc này, tính năng đó mới phát huy tác dụng. Nó sẽ xếp hạng các đoạn văn bản từ gần nhất đến xa nhất, và nhẫn tâm "chặt" bỏ phần đuôi, chỉ nhặt đúng 3 đoạn văn bản có tọa độ gần với câu hỏi nhất.

// Bước 4: Đóng gói và Trả hàng (Return)
// Cuối cùng, nó chuyển 3 cái Vector chiến thắng đó ngược lại thành văn bản tiếng người, đóng gói chúng vào các chiếc hộp có tên là Document, và gán toàn bộ vào biến results.
// [
//     # Kết quả gần nhất (Gần với "phạt đi muộn" nhất)
//     Document(page_content="Đi muộn dưới 15 phút sẽ bị nhắc nhở. Từ 15-30 phút phạt 50k...", metadata={"source": "fpt_policy.txt"}),
    
//     # Kết quả gần thứ hai
//     Document(page_content="Quy định giờ làm việc buổi sáng là 8h00, buổi chiều là 13h30...", metadata={"source": "fpt_policy.txt"}),
    
//     # Kết quả gần thứ ba
//     Document(page_content="Nhân viên vắng mặt không phép sẽ bị trừ lương ngày đó...", metadata={"source": "fpt_policy.txt"})
// ]